In [34]:
import pandas as pd

In [35]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Indexing Pipeline

1. Data loading

In [36]:
df=pd.read_csv('/content/drive/My Drive/Deep Learning IITG/Project/RAG/BBCNews.csv')

In [37]:
columns=['News_ID','Description','Tags']
df.columns=columns
print(df.info())
print(df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2410 entries, 0 to 2409
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   News_ID      2410 non-null   int64 
 1   Description  2410 non-null   object
 2   Tags         2410 non-null   object
dtypes: int64(1), object(2)
memory usage: 56.6+ KB
None
   News_ID                                        Description  \
0        0  chelsea sack mutu  chelsea have sacked adrian ...   
1        1  record fails to lift lacklustre meet  yelena i...   
2        2  edu describes tunnel fracas  arsenals edu has ...   
3        3  ogara revels in ireland victory  ireland flyha...   
4        4  unclear future for striker baros  liverpool fo...   

                                                Tags  
0  sports, stamford bridge, football association,...  
1  sports, madrid, birmingham, france, scotland, ...  
2  sports, derby, brazil, tunnel fracasedu, food,...  
3  sports, bbc, united

In [38]:
from langchain_core.documents import Document
document=[]
for _,row in df.iterrows():
  page_content=row['Description']
  meta_data={'News_ID':row['News_ID'],'Tags':row['Tags']}
  document.append(Document(page_content=page_content,metadata=meta_data))

Data Chunking is the process of spliting words/paragraphs/sentences into either fixed size or adaptive size depending on the type of data so these fixed size text are clubbed together into smaller chunks with overlap between the chunks to ensure continuity

Why chunking:

1. LLM process only context window sized tokens. Anything more than the context length is truncated

2. Lost in the middle problem, if the context lenghts are too large there is problems with the LLM's accuracy when the important information is in the middle of the prompt

3. Faster retrieval, with smaller chunks of the text the search is faster by the retriever

In [41]:
!pip install langchain-text-splitters --quiet
!pip install transformers --quiet
from langchain_text_splitters import CharacterTextSplitter,RecursiveCharacterTextSplitter
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
text_splitter=RecursiveCharacterTextSplitter.from_huggingface_tokenizer(tokenizer,chunk_size=200,chunk_overlap=20)

In [42]:
#Here the data splitting is performed using pretrained tokenizer which performs split based on chunksize
#which are the number of tokens. Why tokenizer is better than charactersplit is because the embedding models
#have token limits or context window size
chunks=text_splitter.split_documents(document)

###Convert the chunks into Embeddings

There are various pretrained Embedding models which can be used like OpenAI or huggingface.

OpenAI uses Cloud API where the text is sent to the API and the vector embeddings are returned. This is done using OpenAI Key.

HuggingFace runs offline locally and is used when data privacy is required and no API cost.

OpenAI performs better than HuggingFace embeddings and are faster to implement

In [43]:
#Using HuggingFace embeddings as it is open source and does not require a API key
!pip install sentence-transformers --quiet

In [44]:
#Sample embeddings on small text
from sentence_transformers import SentenceTransformer

model=SentenceTransformer('all-MiniLM-L6-v2')
sentences=["The weather is lovely today.",
    "It's so sunny outside!",
    "He drove to the stadium.",]
embeddings=model.encode(sentences)
print(embeddings)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[[ 0.01919577  0.12008539  0.15959834 ... -0.00536285 -0.08109502
   0.05021336]
 [-0.01869039  0.0415187   0.07431544 ...  0.00486599 -0.06190438
   0.03187511]
 [ 0.136502    0.08227322 -0.02526163 ...  0.08762044  0.03045843
  -0.01075751]]


In [55]:
#Lets apply embeddings on the entire chunks
chunk_embeddings=model.encode([chunk.page_content for chunk in chunks])

In [57]:
#Lets store these embeddings into a vector database Weaviate
!pip install weaviate-client --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 619.5/619.5 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 2.8 MB/s eta 0:00:00


In [63]:
import weaviate
client = weaviate.connect_to_wcs(cluster_url='https://kuzb1ywqawa8hnml0o2a.c0.asia-southeast1.gcp.weaviate.cloud',
                                 auth_credentials=weaviate.auth.AuthApiKey("T2R1cFVhZnRFNUdHQjJLL19XVWRwOUxwenAxemhiWERQYUdnUlhCRys1TERlM3BZdXlncUE4T3IzVTFrPV92MjAw"))

/tmp/ipykernel_346/3071108937.py:2: DeprecatedWarning: connect_to_wcs is deprecated as of 4.6.2. 
This method is deprecated and will be removed in a future release. Use :func:`connect_to_weaviate_cloud` instead.

  client = weaviate.connect_to_wcs(cluster_url='https://kuzb1ywqawa8hnml0o2a.c0.asia-southeast1.gcp.weaviate.cloud',


In [64]:
print(client.is_ready())

True


In [68]:
import weaviate.classes as wvc
collection=client.collections.create(name='BBCNews',
                                     vectorizer_config=None,
                                     properties=[wvc.config.Property(name='content',data_type=wvc.config.DataType.TEXT),
                                                 wvc.config.Property(name='news_id',data_type=wvc.config.DataType.TEXT),
                                                 wvc.config.Property(name='tags',data_type=wvc.config.DataType.TEXT)])

In [69]:
bbc_collection=client.collections.get('BBCNews')

In [71]:
for doc,emb in zip(document,chunk_embeddings):
  bbc_collection.data.insert(properties={'content':doc.page_content,
                                         'news_id':str(doc.metadata['News_ID']),
                                         'tags':doc.metadata['Tags']},
                             vector=emb)

In [74]:
response=bbc_collection.query.fetch_objects(limit=1,include_vector=True)
print(response.objects[0].vector)

{'default': [-0.015877746045589447, -0.06753850728273392, -0.007331686094403267, -0.01998678781092167, 0.019234351813793182, 0.07181745022535324, 0.04615597799420357, 0.008730483241379261, -0.044498346745967865, -0.05252284184098244, -0.044160082936286926, -0.07087188214063644, -0.012626159936189651, 0.061498600989580154, 0.046308938413858414, 0.017686042934656143, -0.12057129293680191, -0.019655495882034302, -0.030426712706685066, 0.022419894114136696, -0.06935616582632065, 0.035390567034482956, -0.08784355968236923, 0.007204569410532713, -0.042561545968055725, -0.07681476324796677, 0.019978603348135948, 0.061878904700279236, -0.08081082999706268, -0.06983983516693115, 0.004130142275243998, -0.07628079503774643, -0.04746077582240105, -0.02705645188689232, -0.012354712933301926, 0.02482830174267292, -0.045858778059482574, -0.06694598495960236, 0.02971983142197132, 0.0174116063863039, -0.007855674251914024, -0.07476304471492767, -0.03794453293085098, 0.0028792303055524826, -0.0067879185

In [73]:
for obj in response.objects:
  print(obj.properties)

{'content': 'kilroy launches veritas party  exbbc chat show host and east midlands mep robert kilroysilk has said he wants to change the face of british politics as he launched his new party  mr kilroysilk who recently quit the uk independence partysaid our country was being stolen from us by mass immigration he told a london news conference that veritas  latin for truth  would avoid the old parties lies and spin ukip leader roger knapman says he was glad to see the back of mr kilroysilk  mr kilroysilk promised a firm but fair policy on immigration and said they hoped to contest most seats at the forthcoming general election he said veritas would also announce detailed policies on crime tax pensions health and defence over the next few weeks and he announced the party would be holding a leadership election on thursday he is due to announce which constituency he will run in at the next general election  that will come amid speculation he has his sights set on defence secretary geoff hoo

In [75]:
bbc_collection.aggregate.over_all(total_count=True)

AggregateReturn(properties={}, total_count=2410)

###Generation Pipeline

In [87]:
#Semantic search
bbc_collection = client.collections.get('BBCNews') # Ensure bbc_collection is available, though it should be from previous cells
query_vector = model.encode(['latest cricket match results'])
football_res = bbc_collection.query.near_vector(
    near_vector=query_vector[0],
    limit=5,
    return_properties=['content', 'news_id', 'tags']
)

In [88]:
for obj in football_res.objects:
  print(obj.properties)

{'content': 'greek sprinters suspended by iaaf  greek sprinters kostas kenteris and katerina thanou have been suspended after failing to take drugs tests before the athens olympics  athletics ruling body the iaaf said explanations from the pair and their former coach as to why they missed the tests were unacceptable it added that kenteris and thanou had been provisionally suspended pending the resolution of their cases they face twoyear bans if found guilty by the greek athletics federation the suspension also covers the athletes controversial coach christos tzekos kenteris the  olympic m champion and thanou the womens m silver medallist from the same games in sydney also face a criminal hearing in greece over the missed tests they failed to appear to give samples in chicago and tel aviv shortly before the athens games and again in athens on  august the eve of the opening ceremony greek prosecutors have also charged them with faking a midnight motorcycle crash which led to them spendin

In [91]:
#keyword search
bbc_collection = client.collections.get('BBCNews') # Ensure bbc_collection is available, though it should be from previous cells
football_bm25 = bbc_collection.query.bm25(
    query='latest football match results',
    limit=5,
    return_properties=['content', 'news_id', 'tags']
)

In [92]:
for obj in football_bm25.objects:
  print(obj.properties)

{'content': 'wilkinson return unlikely  jonny wilkinson looks set to miss the whole of the  rbs six nations  englands world cupwinning flyhalf said last week he was hoping to recover from his latest injury in time to play some role in the championship but rob andrew coach of wilkinsons club side newcastle said that with only two games left to play wilkinson was unlikely to be fit in time it would be irresponsible to put him straight into a test match andrew told the times wilkinson is recovering from a knee injury which followed longterm neck and arm injuries he has not played for england since the world cup final in november  since when the stuttering world champions have lost nine of their  matches wilkinson is aiming to make his third start to the season in the zurich premiership match against harlequins on  march  that game is the day after england play italy in the six nations and six days before their final match of the championship against scotland we are hoping jonny will be re

In [98]:
import weaviate.classes as wvc

#Metadata filtering
query_vec = model.encode(['latest cricket match results'])
metadata_res = bbc_collection.query.near_vector(
    near_vector=query_vector[0],
    filters=wvc.query.Filter.by_property("tags").equal("cricket"),
    limit=5,
    return_properties=['content', 'news_id', 'tags']
)

In [99]:
for obj in metadata_res.objects:
  print(obj.properties)

{'content': 'sport betting rules in spotlight  a group of mps and peers has called for a tightening of regulations controlling betting on sport  the parliamentary group on betting and gaming held a substantial inquiry into betting last year it followed fears that a massive increase in betting on sport such as that done using the internet and mobile phones has led to more cheating the allparty group recommended  ways to protect punters and improve the integrity of sports betting they include a proposal for raising the maximum jail sentence for gambling cheats above the current two years lord condon head of the international cricket councils anticorruption unit who originally made the call for longer prison sentences said the twoyear penalty was derisory you could get a bigger sentence for failing to pay your hotel bill criminally than you could for corruption in major sports symbolically a higher penalty perhaps as the bill passes through the two houses might be appropriate  the report 

In [112]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass('Enter your OpenAI key:')

Enter your OpenAI key:··········


In [125]:
#Hybrid search
query_string = 'Summarize the football news article from BBC sports section.'
query_vector_for_hybrid = model.encode([query_string])
retriever_query = (
    bbc_collection.query.hybrid(
        query=query_string, # For BM25 part of hybrid search
        vector=query_vector_for_hybrid[0], # For vector search part of hybrid search
        alpha=0.6,
        limit=5,
        return_properties=['content', 'news_id', 'tags']
    )
)

results=retriever_query
docs=[item.properties['content'] for item in results.objects]

content='\n\n'.join(docs)

prompt=f"""
Use the following content to answer the question.
Context:{content}
Question:{query_string}
Answer:
"""

!pip install langchain_openai --quiet
!pip install langchain_core --quiet
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
llm=ChatOpenAI(model_name='gpt-3.5-turbo',temperature=0.2)
response=llm.invoke(input=[HumanMessage(content=prompt)])
print('Answer:',response.content)

Answer: The article discusses the split between developers Sports Interactive and Eidos, resulting in the creation of a new football management game called Football Manager. The game is described as the spiritual successor to the Championship Manager series, offering a realistic and immersive experience with improvements in various areas such as the match engine and user interface. The game allows players to manage teams from various leagues and nations, with a focus on tactics, transfers, and training. Overall, Football Manager is praised as a complex and engaging game that will appeal to both fans of the previous CM games and newcomers to the genre.


In [135]:
!pip install streamlit --quiet
!pip install pyngrok --quiet

In [146]:
%%writefile app.py
import streamlit as st

st.title("BBC News RAG Chatbot")

if "history" not in st.session_state:
    st.session_state.history = []

query = st.text_input("Ask a question about BBC News:")

if query:

    query_vector_for_hybrid = model.encode([query])[0]
    retriever_query = (
        bbc_collection.query.hybrid(
            query=query, # Use user_query for BM25 part too
            vector=query_vector_for_hybrid, # For vector search part of hybrid search
            alpha=0.6,
            limit=5,
            return_properties=['content', 'news_id', 'tags']
        )
    )

    results=retriever_query
    docs=[item.properties['content'] for item in results.objects]

    content='\n\n'.join(docs)

    prompt=f"""
    Use the following content to answer the question.
    Context:{content}
    Question:{query}
    Answer:
    """

    response=llm.invoke(input=[HumanMessage(content=prompt)])
    st.session_state.history.append((query, response.content))

# Display chat history
for q, a in st.session_state.history:
    st.write(f"**You:** {q}")
    st.write(f"**Bot:** {a}")

Overwriting app.py


In [138]:
!ngrok authtoken 3AzAaXJZhbcxvC6TxpTpETxSESS_81oGTa1WBnNQyTUinUTQ7

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [145]:
from pyngrok import ngrok

# Set up a tunnel to the streamlit port 8501
public_url = ngrok.connect(8501)
print("Your Streamlit app is live at:", public_url)

# Run Streamlit
!streamlit run app.py &>/dev/null &

Your Streamlit app is live at: NgrokTunnel: "https://unidentical-philomena-nonmetaphysically.ngrok-free.dev" -> "http://localhost:8501"
